# AudioSmith AI 🎙️✨ - DeepFilterNet Fine-Tuning

This notebook fine-tunes the official DeepFilterNet model using the AudioSmith AI machine learning pipeline. It is fully self-contained and orchestrates the dataset preparation, training, evaluation, and export steps automatically.

### ⚠️ Instructions
1. **Enable GPU**: Ensure the Accelerator in the right panel is set to `GPU P100` or `GPU T4x2`.
2. **Internet**: Ensure Internet is toggled **ON** in the right panel so the repository and datasets can be downloaded.
3. **Run All**: Click **Run All** to start the pipeline.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not enabled! Training will be extremely slow. Please enable a GPU in the Kaggle settings.")

## 1. Clone Repository & Install Dependencies
We clone the AudioSmith repository to use the existing `finetune.py`, `evaluate.py`, and `dataset.py` logic.

In [ ]:
import os
import sys
import subprocess
import re

REPO_URL = "https://github.com/yourusername/AudioSmith.git" # <-- Update this to your repo URL if needed
REPO_DIR = "/kaggle/working/AudioSmith"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=False)
else:
    print("Repository already cloned.")

os.chdir(REPO_DIR)

print("Installing Rust (required for DeepFilterNet on Python 3.12+)...")
subprocess.run('curl --proto \'=https\' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y', shell=True, check=True)
os.environ['PATH'] += ':/root/.cargo/bin'

print("Installing dependencies...")
with open('ml/pyproject.toml', 'r') as f:
    content = f.read()
deps_match = re.search(r'dependencies\s*=\s*\[(.*?)\]', content, re.DOTALL)
if deps_match:
    deps_raw = deps_match.group(1)
    deps = [line.split('"')[1] for line in deps_raw.split('\n') if '"' in line and not line.strip().startswith('#')]
    print('Found dependencies:', deps)
    subprocess.run([sys.executable, '-m', 'pip', 'install'] + deps, check=True)
else:
    print('Could not parse dependencies')


## 2. Prepare Datasets
AudioSmith's `download_assets.sh` script will fetch LibriSpeech (train-clean-100), MUSAN, and VoiceBank-DEMAND. We configure `DATASET_ROOT` so the datasets align with `ml/configs/train_config.yaml`.

In [ ]:
os.chdir(REPO_DIR)
!chmod +x scripts/download_assets.sh
!export DATASET_ROOT=./ml/data && ./scripts/download_assets.sh

## 3. Fine-Tune DeepFilterNet
We run the fine-tuning pipeline. DeepFilterNet's official weights are automatically downloaded internally. The trainer tracks progress via MLflow and saves checkpoints.

In [ ]:
# Optional: Override config to run faster for a demo, or keep default config.
# For example, you can edit ml/configs/train_config.yaml here if needed.

os.chdir(REPO_DIR)
!export PYTHONPATH=. && python ml/scripts/finetune.py

## 4. Evaluate Fine-Tuned Model
Evaluate the newly fine-tuned model against the official DeepFilterNet model on the VoiceBank-DEMAND dataset.

In [ ]:
os.chdir(REPO_DIR)
!export PYTHONPATH=. && python ml/scripts/evaluate.py --checkpoint ml/checkpoints/best_model.pt

## 5. Export Model to ONNX
Export the best checkpoint to an ONNX graph for potential external deployment.

In [ ]:
os.chdir(REPO_DIR)
!mkdir -p ml/exports
!export PYTHONPATH=. && python ml/scripts/export_model.py --model deepfilternet --checkpoint ml/checkpoints/best_model.pt --output ml/exports/fine_tuned.onnx --format onnx

## 6. Package and Export Artifacts
We zip the `checkpoints`, `exports`, and `mlruns` folders so they can be downloaded easily from the Kaggle Output panel.

In [ ]:
os.chdir(REPO_DIR)
!zip -r /kaggle/working/AudioSmith_Finetuned_Model.zip ml/checkpoints ml/exports ml/mlruns
print("\n=========================================================")
print("✅ SUCCESS!\n")
print("Your fine-tuned model has been packaged into `AudioSmith_Finetuned_Model.zip`.")
print("\n📥 INSTRUCTIONS FOR DEPLOYMENT:")
print("1. Look at the 'Output' panel on the right side of this Kaggle notebook.")
print("2. Download `AudioSmith_Finetuned_Model.zip`.")
print("3. Extract the ZIP file locally.")
print("4. Copy `best_model.pt` into your local AudioSmith repository (e.g., `ml/checkpoints/best_model.pt`).")
print("5. Configure your AudioSmith backend (or `.env`) to load this checkpoint.")
print("6. Enjoy your custom fine-tuned model in production without any backend code changes!")
print("=========================================================")